# EDA: O que 1000 vendas de e-commerce revelam sobre comportamento do consumidor?

O varejo online cresce continuamente, entender o comportamento dos clientes é essencial para estrategia de negócio. Este projeto analisa **1000 pedidos de um e-commerce simulado**, explorando padrões de consumo, perfil do consumidor, preferencia de pagamentos e níveis de satisfação.

O dataset cobre um período de 1 ano (03/2024 - 03/2025) e inclui dados de clientes, produtos, pagamentos e avaliações.

**Github**: [Github](https://github.com/victorxavier01)

**LinkedIn**: [LinkedIn](https://www.linkedin.com/in/victor-xavier-89a339378/)

**Medium**: [Medium](https://medium.com/@lvsxmk23)

**Objetivos:**

- **Quanto a empresa faturou e como foi a evolução ao longo do tempo?**
- **Perfil demográfico: gênero, faica etária e localizção**
- **Analisar o review_score e suas correlações**
- **Como segmentar os clientes para campanhas de marketing.**

**O que será usado:**
- **Pandas**
- **Numpy**
- **Plotly, seaborn, matplotlib para visualização**
- **Scikit-learn para regressão linear**


## Sumário

- [1. Carregamento e limpeza](#limpeza)
- [2. KPIs e tendencias](#kpis)
- [3. Perfil do cliente](#cliente)
- [4. Satisfação e review](#satisfacao)
- [5. Segmentação (RFM + Clustering)](#segmentacao)
- [6. Insights e recomendações](#insights)
- [7. Encerramento](#encerramento)

In [ ]:
# Importações
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Configurações de estilo do plotly
px.defaults.template = 'plotly_white'
px.defaults.width = 900
px.defaults.height = 500

## 1. Carregamento e limpeza
<div id="limpeza"></div>

In [ ]:
# O dataset já se encontra limpo desde a fonte, mas irei realizar uma limpeza por motivos de prática.

df = pd.read_csv('synthetic_online_retail_data.csv')

print(f"Linhas: {len(df)}, Colunas: {len(df.columns)}")

print('\nPrimeiras 5 linhas')
df.head()


In [ ]:
# Como podemos ver, temos uma coluna de data! Então é necessário verificar o tipo dessa coluna para sabermos se há uma string ou data de fato. 
# Além disso, podemos fazer algumas feature engineering com quantidade e preço, data e idade!

print(df['order_date'].dtype)

In [ ]:
# Corrigindo tipo de data

df['order_date'] = pd.to_datetime(df['order_date'])

print(f"Novo tipo da coluna data: {df['order_date'].dtype}\n")

# Criando coluna receita
df['receita'] = df['quantity'] * df['price']

# Features temporais, a separação vai nos ajudar a plotar gráficos e treinar modelos posteriormente
df['mes'] = df['order_date'].dt.month
df['trimestre'] = df['order_date'].dt.quarter

# Faixas etárias, agrupar por faixas etárias também vai nos ajudar a plotar gráficos e treinar modelos.
df['faixa_etaria'] = pd.cut(
    df['age'],
    bins=[0, 30, 45, 60, 100],
    labels=['18-30', '31-45', '46-60', '60+'])

# Visualizando o estado atual do DF.
df.head()


In [ ]:
# Tratando valores nulos. 
# Vamos preencher os valores nulos com a mediana, que é mais segura do que a média nesse caso.
df['review_score'] = df['review_score'].fillna(df['review_score'].median())

# Generos ausentes
df['gender'] = df['gender'].fillna('Desconhecido')

df.head(3)

## 2. KPIs e tendencias
<div id="kpis"></div>

In [ ]:
# KPIs
receita_total = df['receita'].sum()
ticket_medio = df['receita'].mean()
total_pedidos = len(df)
media_reviews = df['review_score'].mean()
clientes_unicos = df['customer_id'].nunique()

# Cards dos KPIs
fig, axes = plt.subplots(1, 5, figsize=(25,6))
kpis = [
    (f'R${receita_total/1000:,.1f}K', 'Receita Total'),
    (f'R${ticket_medio:,.0f}', 'Ticket Médio'),
    (f'{total_pedidos}', 'Pedidos'),
    (f'{media_reviews:.1f}/5.0', 'Média Reviews'),
    (f'{clientes_unicos}', 'Clientes')
]

# Plotando KPIs
for ax, (valor, titulo) in zip(axes, kpis):
    ax.text(0.5, 0.6, valor, fontsize=28, fontweight='bold', ha='center')
    ax.text(0.5, 0.3, titulo, fontsize=22, ha='center', color='gray')
    ax.set_axis_off()

plt.tight_layout()


In [ ]:
# Plotando evolução das vendas

vendas_mensais = df.groupby('mes')['receita'].sum().reset_index() # Resetando o index para voltar a ser um DF normal

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=vendas_mensais['mes'],
    y=vendas_mensais['receita'],
    name='Receita',
    mode='lines',
    line_shape='spline',
    fill='tozeroy'
))
fig.update_layout(
    title='Evolução da Receita Mensal',
    xaxis_title='Mês',
    yaxis_title='Receita (R$)',
    xaxis_tickvals=[1,2,3,4,5,6,7,8,9,10,11,12],
    xaxis_ticktext=['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
)

fig.show()

In [ ]:
# Receita por categoria
receita_cat = df.groupby('category_name')['receita'].sum().sort_values(ascending=False).reset_index()

fig = go.Figure(go.Bar(
    x=receita_cat['category_name'],
    y=receita_cat['receita'],
    marker_color='#4a90d9',
    text=[f'R${v/1000:.1f}K' for v in receita_cat['receita']],
    textposition='outside'
))
fig.update_layout(
    title='Receita Total por Categoria',
    xaxis_title='Categoria',
    yaxis_title='Receita (R$)',
    xaxis_tickangle=-45
)

fig.show()

# Formas de pagamento

formas_pagamento = df['payment_method'].value_counts().reset_index()
formas_pagamento.columns = ['payment_method', 'count']

fig = go.Figure(go.Pie(
    labels=formas_pagamento['payment_method'],
    values=formas_pagamento['count'],
    hole=0.4
))
fig.update_layout(title='Distribuição por Forma de Pagamento')

fig.show()

    

## 3. Perfil Demográfico
<div id="cliente"></div>

In [ ]:
# Distribuição por genero

fig, axes = plt.subplots(1, 2, figsize=(14,6))

# Genero
count_generos = df['gender'].value_counts()
axes[0].pie(count_generos.values, labels=count_generos.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Distribução por Genero')

# Faixa etária
count_idade = df['faixa_etaria'].value_counts()
axes[1].bar(count_idade.index, count_idade.values, color='#4a90d9')
axes[1].set_title('Distribuição por Faixa Etária')
axes[1].set_xticklabels(count_idade.index, rotation=0)

plt.tight_layout()
plt.show()

#### Interpretando

Não há diferença significativa entre geneneros ou idade. Os clientes apresentam uma distribuição bem homogenea.

## 4. Satisfação
<div id="satisfacao"></div>

In [ ]:
# Reviews
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=df['review_score'],
    nbinsx=10,
    marker_color='#2ecc71',
    hovertemplate='%{x:.1f} stars: %{count} pedidos<br><extra>Distribuição</extra>'
))
fig.update_layout(
    title='Distribuição das Notas de Avaliação',
    xaxis_title='Nota',
    yaxis_title='Quantidade de Pedidos'
)

fig.show()

In [ ]:
cat_review = df.groupby('category_name')['review_score'].mean().sort_values(ascending=False).reset_index()

fig = go.Figure(go.Bar(
    x=cat_review['category_name'],
    y=cat_review['review_score'],
    marker=dict(
        color=['#e74c3c' if x < 3.8 else '#f39c12' if x < 4.0 else '#2ecc71' for x in cat_review['review_score']]
    ),
    text=[f'{x:.2f}' for x in cat_review['review_score']],
    textposition='outside'
))
fig.update_layout(
    title='Média de Review por Categoria',
    xaxis_title='Categoria',
    yaxis_title='Nota Média',
    yaxis=dict(range=[0, 5.5]),
    xaxis_tickangle=-45
)

# CorreÃ§Ã£o do alpha -> opacity
fig.add_hline(y=4.0, line_dash='dash', line_color='gray', opacity=0.7)

fig.show()


In [ ]:
# Scatter price vs quantity colorido por review

fig = go.Figure(go.Scatter(
    x=df['price'],
    y=df['quantity'],
    mode='markers',
    marker=dict(
        color=df['review_score'],
        colorscale='Hot',
        size=8,
        showscale=True,
        colorbar=dict(title='Review')
    ),
    text=df['category_name'],
    hovertemplate='Preço: R$%{x:.2f}<br>Quantidade: %{y}<br>Categoria: %{text}<br>Review: %{marker.color:.1f}'
))
fig.update_layout(
    title='Preço vs Quantidade (cor: Review Score)',
    xaxis_title='Preço (R$)',
    yaxis_title='Quantidade Comprada'
)

fig.show()

### Interpretando
As reviews apresentam notas bem consistentes entre as categorias. Sports & Outdoors lidera levemente as avaliações, sendo a única acima da média de todas as categorias.

## 5. Segmentação
<div id="segmentacao"></div>

In [ ]:
# RFM
# Obs: cada cliente tem exatamente 1 pedido nesta base (1000 clientes / 1000 pedidos),
# entao Frequency = 1 para todos; a segmentacao fica efetivamente em Recencia + Monetario.
ultima_compra = df.groupby('customer_id')['order_date'].max()
data_ref = pd.Timestamp('2025-03-31')
recencia_dias = (data_ref - ultima_compra).dt.days

rfm = df.groupby('customer_id').agg(
    monetario=('receita', 'sum'),
    frequencia=('order_date', 'count')
).reset_index()

# Alinhar pela chave customer_id (evita atribuicao por posicao)
rfm['recencia'] = rfm['customer_id'].map(recencia_dias)

print("Top 5 maiores clientes:")
rfm.nlargest(5, 'monetario')

In [ ]:
# Clusterização
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X_rfm = rfm[['recencia', 'frequencia', 'monetario']].values

# Escalonando as features (para não haver viés)
scaler_rfm = StandardScaler()
X_rfm_scaled = scaler_rfm.fit_transform(X_rfm)

# Descobrir K ideal com Elbow
inercias = []
for i in range(1, 11):
    kmeans_i = KMeans(n_clusters=i, init='k-means++', n_init=10, max_iter=300, random_state=42)
    kmeans_i.fit(X_rfm_scaled)
    inercias.append(kmeans_i.inertia_)

# K escolhido a partir do cotovelo (elbow) do grafico plotado abaixo
k_selecionado = 4

# plotando o elbow
grafico_elbow = go.Figure()
grafico_elbow.add_trace(go.Scatter(
    x=list(range(1, 11)),
    y=inercias,
    mode='lines+markers',
    marker=dict(size=8, color='#636EFA'),
    line=dict(color='#636EFA', width=3)
))
grafico_elbow.add_hline(y=inercias[k_selecionado - 1], line_dash='dot', line_color='gray')
grafico_elbow.update_layout(
    title='Elbow method para achar o melhor K',
    xaxis_title='Número de Clusters (K)',
    yaxis_title='Inercia (WCSS)',
    template='plotly_white'
)
grafico_elbow.show()

# Aplicar K
kmeans_final = KMeans(n_clusters=k_selecionado, init='k-means++', n_init=10, max_iter=300, random_state=42)
rfm['cluster'] = kmeans_final.fit_predict(X_rfm_scaled)

# Perfil de cada cluster
perfil_clusters = rfm.groupby('cluster').agg(
    qtd_clientes=('customer_id', 'count'),
    rec_media=('recencia', 'mean'),
    freq_media=('frequencia', 'mean'),
    monetario_medio=('monetario', 'mean')
).round(2)

print(f'\nPERFIL DOS {k_selecionado} CLUSTERS\n')
print(perfil_clusters)

# Plotando os clusters
grafico_scatter = px.scatter(
    rfm,
    x='recencia',
    y='monetario',
    color='cluster',
    color_discrete_sequence=px.colors.qualitative.Set1,
    title=f'Clusters de Clientes (K={k_selecionado}): Recência e Monetário',
    labels={'recencia': 'Recência (dias)', 'monetario': 'Receita Total (R$)'},
    template='plotly_white'
)
grafico_scatter.show()

In [ ]:
# Grafico de bar, gasto medio por cluster
px.bar(
    x=['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3'],
    y=rfm.groupby('cluster')['monetario'].mean().values,
    labels={'x': 'Cluster', 'y': 'Gasto Medio (R$)'},
    title='Gasto Medio por Cluster'
).update_layout(template='plotly_white')



In [ ]:
# Grafico de pizza por distribuicao
px.pie(
    values=rfm.groupby('cluster').size().values,
    names=['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3'],
    title='Distribuicao por Cluster'
).update_layout(template='plotly_white')

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Hipótese: preço e quantidade explicam a satisfação do cliente?
# Alvo: review_score (1-5). Features: price e quantity.
# O dropna deixa esta celula independente da ordem de execucao (a base tem 201 reviews nulas).
df_sat = df[['price', 'quantity', 'review_score']].dropna()
X_sat = df_sat[['price', 'quantity']].copy()
y_sat = df_sat['review_score'].copy()

# Treino e teste (80/20)
X_treino, X_teste, y_treino, y_teste = train_test_split(X_sat, y_sat, test_size=0.2, random_state=42)

# Treinar modelo
modelo_sat = LinearRegression()
modelo_sat.fit(X_treino, y_treino)

# Prever e analisar
y_pred = modelo_sat.predict(X_teste)
r2 = r2_score(y_teste, y_pred)
rmse = np.sqrt(mean_squared_error(y_teste, y_pred))

print(f'Amostra: {len(df_sat)} pedidos (sem reviews nulas)')
print(f'R2 Score: {r2:.2%}')
print(f'RMSE: {rmse:.2f} pontos')
print(f'Coef. Preço: {modelo_sat.coef_[0]:.4f}')
print(f'Coef. Quantidade: {modelo_sat.coef_[1]:.4f}')

# Plotando real e previsto
grafico_sat = px.scatter(
    x=y_teste.values,
    y=y_pred,
    opacity=0.6,
    title='Satisfação: Nota Real vs Prevista (Regressão Linear)',
    labels={'x': 'Nota Real', 'y': 'Nota Prevista'}
)
grafico_sat.add_trace(go.Scatter(
    x=[y_sat.min(), y_sat.max()],
    y=[y_sat.min(), y_sat.max()],
    mode='lines',
    line=dict(color='red', dash='dash'),
    name='Linha Ideal'
))
grafico_sat.update_layout(template='plotly_white')
grafico_sat.show()


### Interpretando

O **R² próximo de zero** indica que preço e quantidade **não explicam** a variação da satisfação nesta base, um resultado negativo, e ele é reportado como tal. Em dados reais, investigaríamos variáveis adicionais (experiência de entrega, atendimento, qualidade do produto) ou trocaríamos o alvo do modelo (ex: receita futura, churn).


## 6. Insights
<div id="insights"></div>

1 - Eletronics é a categoria mais rentável, merecendo mais investimentos em estoque e marketing.\
2 - A satisfação do cliente está em nível moderado (3,5/5), com oportunidades de melhoria.\
3 - A segmentação RFM permite marketing direcionado

## 7. Encerramento
<div id="encerramento"></div>

Por se tratar de um dataset artificial, poucos insights puderam ser obtidos, pois todas as features apresentam valores semelhantes. Caso se interesse mais pelo meu trabalho, você pode acessar as redes sociais abaixo:

**Github**: [Github](https://github.com/victorxavier01)

**LinkedIn**: [LinkedIn](https://www.linkedin.com/in/victor-xavier-89a339378/)

**Medium**: [Medium](https://medium.com/@lvsxmk23)